# Y = ReLU(XW + b): JAX → jaxpr → StableHLO → HLO on TPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/jax_tpu_relu_linear.ipynb)

`jax.jit`이 식 하나를 **추적 → jaxpr → StableHLO → XLA TPU 백엔드(HLO) → 실행**으로 내리는 과정을 단계별로 출력한다. TPU에서만 보이는 타일 레이아웃 `T(8,128)`, VMEM 메모리 공간 `S(1)`, DMA `copy-start/copy-done`, 패딩된 버퍼 크기를 확인한다.

**런타임 설정**: 메뉴 `런타임 → 런타임 유형 변경 → TPU` (v5e 또는 v6e).

문서: 저장소의 `docs/example-jax-tpu-v5e.md` 의 각 절 번호와 셀 제목이 대응한다.

## 0. 환경 확인

In [ ]:
import jax, jax.numpy as jnp, re
d = jax.devices()[0]
assert d.platform == "tpu", f"런타임 유형을 TPU로 바꾸세요 (현재: {d.platform})"
print("jax", jax.__version__)
print("devices:", jax.devices())
print("kind:", d.device_kind, "| device_count:", jax.device_count())
ms = d.memory_stats()
print("HBM limit: %.2f GiB" % (ms["bytes_limit"] / 2**30))

In [ ]:
def f(x, w, b):
    return jax.nn.relu(x @ w + b)

x = jnp.ones((16, 8), jnp.float32)
w = jnp.ones((8, 4), jnp.float32)
b = jnp.ones((4,), jnp.float32)

def strip_metadata(hlo_text):
    # 가독성을 위해 metadata={...} 제거
    return re.sub(r", metadata=\{[^}]*\}", "", hlo_text)

## 1. 추적 → jaxpr

`dot_general`의 `dimension_numbers`가 contracting 축, `broadcast_in_dim`이 명시적 노드, `relu`는 `custom_jvp_call`로 감싸여 있다.

In [ ]:
print(jax.make_jaxpr(f)(x, w, b))

## 2. lowering → StableHLO

플랫폼 독립 IR. CPU에서 뽑아도 글자 하나 다르지 않다. 여기서 JAX의 역할이 끝난다.

In [ ]:
lowered = jax.jit(f).lower(x, w, b)
print(lowered.as_text())

## 3. XLA TPU 백엔드 → 최적화된 HLO

읽을 것:

- `fusion(...) kind=kOutput` **하나**에 `convolution + broadcast + add + maximum`이 다 들어감
- `dot`이 `convolution(...), dim_labels=bf_io->bf`로 바뀜
- 레이아웃 `{0,1:T(8,128)}`: minor-to-major `{0,1}`(열 우선) + 8×128 타일
- `S(1)` = VMEM, `copy-start`/`copy-done` = HBM→VMEM 비동기 DMA, `cross_program_prefetch_index`
- `backend_config`의 `estimated_cycles`, `used_scoped_memory_configs`

In [ ]:
compiled = lowered.compile()
print(strip_metadata(compiled.as_text()))

### 3-6. 비용·메모리 분석

`argument_size_in_bytes`가 실제 데이터(656 B)보다 훨씬 큰 이유는 타일 패딩이다.

In [ ]:
print(compiled.cost_analysis())
print(compiled.memory_analysis())

real = sum(a.size * a.dtype.itemsize for a in (x, w, b))
print(f"\n실제 인자 바이트: {real}  vs  XLA argument_size_in_bytes: {compiled.memory_analysis().argument_size_in_bytes}")

### 3-7. bf16으로 바꾸면

레이아웃에 `(2,1)`(bf16 2개를 32비트 워드에 패킹)이 붙고 VMEM 스크래치가 절반이 된다.

In [ ]:
xb, wb, bb = (t.astype(jnp.bfloat16) for t in (x, w, b))
cb = jax.jit(f).lower(xb, wb, bb).compile()
txt = strip_metadata(cb.as_text())
print(txt.splitlines()[0])                         # entry_computation_layout
print(re.search(r'backend_config=\{.*\}', txt).group(0)[:400])

## 5. 실행과 비동기 디스패치

`jax.Array`는 future다. 값을 읽을 때만 기다린다.

In [ ]:
y = jax.jit(f)(x, w, b)
print(type(y), y.shape, y.dtype)
print(y.sharding)
print(y[:2])

In [ ]:
import time
a = jnp.ones((4096, 4096), jnp.bfloat16)
(a @ a).block_until_ready()                        # 웜업 (컴파일 포함)

t0 = time.perf_counter(); z = a @ a; t1 = time.perf_counter()
z.block_until_ready();     t2 = time.perf_counter()
print(f"dispatch only            : {(t1 - t0) * 1e6:8.1f} us")
print(f"until block_until_ready(): {(t2 - t0) * 1e3:8.3f} ms")
flops = 2 * 4096**3
print(f"≈ {flops / (t2 - t0) / 1e12:.0f} TFLOP/s")

## 6. 큰 행렬로 보는 타일링

2048×2048 bf16이면 레이아웃이 `{1,0}`으로 바뀌고, MXU emitter·루프 구조(`iteration_bounds`)·VMEM 사용량이 달라진다. arithmetic intensity를 계산해 compute-bound인지 본다.

In [ ]:
def g(a, b):
    return jax.nn.relu(a @ b)

A = jnp.ones((2048, 2048), jnp.bfloat16)
B = jnp.ones((2048, 2048), jnp.bfloat16)
cg = jax.jit(g).lower(A, B).compile()
txt = strip_metadata(cg.as_text())
print(txt[txt.index("ENTRY"):])

ca = cg.cost_analysis()
print("\nflops:", ca["flops"], "| bytes:", ca["bytes accessed"],
      "| arithmetic intensity: %.0f FLOPs/byte" % (ca["flops"] / ca["bytes accessed"]))

## 7. 직접 써 보는 Pallas 커널 (선택)

XLA 대신 커널을 직접 쓰면 VMEM 블록과 그리드를 사용자가 지정한다. GPU 문서의 Triton 커널에 대응한다. Colab 런타임의 jax/libtpu 버전 조합에 따라 Mosaic 컴파일이 실패할 수 있어, 그 경우 interpret 모드로 대체한다.

In [ ]:
from jax.experimental import pallas as pl

def relu_bias_kernel(xw_ref, b_ref, o_ref):
    o_ref[...] = jnp.maximum(xw_ref[...] + b_ref[...], 0.0)

xw = x @ w                                          # [16, 4]
call = lambda interpret: pl.pallas_call(
    relu_bias_kernel,
    out_shape=jax.ShapeDtypeStruct(xw.shape, xw.dtype),
    interpret=interpret,
)(xw, b[None, :])

try:
    out = call(interpret=False)                     # Mosaic 컴파일러로 TPU 커널 생성
    print("Pallas on TPU (Mosaic): OK")
except Exception as e:
    # Colab 런타임의 jax와 libtpu 버전이 어긋나면 Mosaic 직렬화 오류가 난다 (예: jax 0.7.2 + libtpu 0.0.21.1).
    # 그 경우 interpret 모드로 같은 커널을 실행한다. 커널 의미는 같지만 TPU에서 돌지는 않는다.
    print("Pallas on TPU 실패:", type(e).__name__, str(e).splitlines()[0][:100])
    out = call(interpret=True)
    print("Pallas interpret mode: OK")
print("matches jax.jit result:", bool(jnp.allclose(out, y)))
